> ## SOLUTION / ANSWER KEY &mdash; Lab 4.7
> This is the **completed** notebook (all `___` blanks filled). For the student version, open
> [`../lab-07-feature-extraction-head.ipynb`](../lab-07-feature-extraction-head.ipynb). Trainer use &mdash; or self-check after you've tried it yourself.

# Lab 4.7 &mdash; Transfer Learning: Frozen Features + a Trainable Head

**Level:** Intermediate &nbsp;|&nbsp; **Est. time:** 30 min &nbsp;|&nbsp; **Day 2 &middot; Module 4 &mdash; Pre-trained Models & Fine-tuning**

### What you'll do
- Use a frozen feature extractor to turn text into vectors
- Train only a small classifier head on top
- See why this is fast, data-efficient transfer learning

> **How this lab works (experiential flow):** read the **Concept**, run the **Demo** to see it work, then complete **Your Turn** by replacing every `___` placeholder. Run the **grader** cell at the end &mdash; it prints `[PASS]` / `[FAIL]` / `[TODO]` and a final `Score`. Aim for a full score.

**Reference:** [Module 4 slides &mdash; Transfer learning](../../presentation/day2-module4-pretrained-models-and-fine-tuning.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 4 labs](./index.html)

In [ ]:
# Setup -- run me first
import os
WORK = "/tmp/biaa-lab-04-07"
os.makedirs(WORK, exist_ok=True)
print("Working dir:", WORK)

## Concept
**Fine-tuning** in its cheapest form: keep a powerful **frozen feature extractor** and train
only a small **head**. Here TF-IDF is the frozen extractor and `LogisticRegression` is the head.
(In the sandbox you swap TF-IDF for a frozen BERT &mdash; the *shape* is identical; see the
optional cell.)

In [ ]:
# DEMO
# A tiny labelled sentiment dataset (1 = positive, 0 = negative)
SENT = [
    ("i love this it is great", 1),
    ("a great and wonderful film", 1),
    ("truly wonderful i love it", 1),
    ("excellent and brilliant work", 1),
    ("the best most brilliant story", 1),
    ("i love how great it is", 1),
    ("wonderful excellent and great fun", 1),
    ("a brilliant and great success", 1),
    ("great fun i really love it", 1),
    ("the best film wonderful and brilliant", 1),
    ("excellent great and lovely work", 1),
    ("i love this brilliant great film", 1),
    ("wonderful great and the best", 1),
    ("so good i love it great", 1),
    ("i hate this it is terrible", 0),
    ("a terrible and awful film", 0),
    ("truly awful i hate it", 0),
    ("boring and terrible work", 0),
    ("the worst most boring story", 0),
    ("i hate how bad it is", 0),
    ("awful boring and dull mess", 0),
    ("a terrible and bad failure", 0),
    ("boring mess i really hate it", 0),
    ("the worst film awful and boring", 0),
    ("terrible bad and dull work", 0),
    ("i hate this awful boring film", 0),
    ("awful terrible and the worst", 0),
    ("so bad i hate it terrible", 0),
]
texts  = [t for t, y in SENT]
labels = [y for t, y in SENT]
from sklearn.model_selection import train_test_split
Xtr_text, Xval_text, ytr, yval = train_test_split(texts, labels, test_size=0.25,
                                                   random_state=0, stratify=labels)
print("train:", len(Xtr_text), "| val:", len(Xval_text))

## Your Turn
Fit the frozen extractor on the training text, transform both splits, train the head, score it.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

def run():
    vec = TfidfVectorizer()
    Xtr = vec.fit_transform(Xtr_text)
    Xval = vec.transform(Xval_text)
    head = LogisticRegression(max_iter=1000)
    head.fit(Xtr, ytr)
    return accuracy_score(yval, head.predict(Xval))

try: print('val accuracy:', round(run(), 3))
except Exception as e: print('Fill the blanks, then re-run.', type(e).__name__)

In [ ]:
# === Auto-grader: run after filling the blanks above ===
_results = []
def _rec(label, status, extra=""):
    _results.append(status); print(f"[{status}] {label}" + (f" -- {extra}" if extra else ""))
def expect(label, got, want):
    if got == "___" or got is None: _rec(label, "TODO")
    elif got == want: _rec(label, "PASS")
    else: _rec(label, "FAIL", f"got {got!r}")
def expect_true(label, fn):
    try: _rec(label, "PASS" if fn() else "FAIL")
    except Exception as e: _rec(label, "TODO", type(e).__name__)

expect_true("pipeline returns an accuracy", lambda: isinstance(run(), float))
expect_true("val accuracy >= 0.8 (transfer learning works)", lambda: run() >= 0.8)

_p = _results.count("PASS")
print(f"\nScore: {_p}/{len(_results)}")
print("All checks passed -- lab complete!" if _p == len(_results) else "Keep going: fill the blanks marked ___ and re-run.")

## Optional &mdash; the real thing with Hugging Face (not graded)
Swap TF-IDF for a REAL frozen transformer as the feature extractor, then train the same head. Safe to skip &mdash; it needs `pip install transformers torch` and a one-time model
download. If `transformers` is not installed (or offline), the cell prints a note and moves on.
**Real BERT fine-tuning is the managed-sandbox target; the graded steps above run offline so the
lab always works.**

In [ ]:
try:
    from transformers import pipeline
    import numpy as np
    fe = pipeline("feature-extraction", model="prajjwal1/bert-tiny")
    def embed(t):
        v = np.array(fe(t)[0])      # (tokens, dim)
        return v.mean(axis=0)       # mean-pool to one vector
    print("bert-tiny embedding dim:", embed("a great film").shape)
    print("Same recipe: frozen transformer features -> a small trainable head.")
except Exception as e:
    print("transformers not available -- the TF-IDF version above already taught the pattern.", type(e).__name__)

---
### Key takeaway
Frozen extractor + trainable head = transfer learning. It needs little data and trains in seconds &mdash; the workhorse before you ever unfreeze the big model.

[&#8592; All Module 4 labs](./index.html) &nbsp;&middot;&nbsp; [Module 4 slides](../../presentation/day2-module4-pretrained-models-and-fine-tuning.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>